# 05 — Model Definition: EfficientNet-B0 U-Net
 **Purpose:** Define the segmentation model architecture, verify the forward pass,
 and save an architecture summary.

 ### Key design rules:
 - Output is **raw logits** — sigmoid is NOT applied inside the model.
 - Loss function (`BCEWithLogitsLoss`) is applied during training, not here.
 - No training occurs in this notebook.
 - `segmentation_models_pytorch` (smp) is used if installed.
   If missing, a minimal fallback U-Net is defined for forward-pass testing only.
   The fallback is NOT a replacement for smp — install smp before training.

# 1  Imports

In [7]:
import sys
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path

print(f"Python  {sys.version}")
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Python  3.13.1 (tags/v3.13.1:0671451, Dec  3 2024, 19:06:28) [MSC v.1942 64 bit (AMD64)]
PyTorch 2.6.0+cu124
CUDA available: True


# 2  Constants

In [8]:
OUTPUT_ROOT  = Path(r"D:\DIABETES\Segmentation_Dataset\Segmentation_Branch_Outputs")
REPORTS_DIR  = OUTPUT_ROOT / "07_reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

ARCH_SUMMARY_PATH = REPORTS_DIR / "05_model_architecture_summary.txt"

IMG_SIZE    = 384
IN_CHANNELS = 3
NUM_CLASSES = 1    # binary segmentation

# 3  Device

In [9]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


Device: cuda


# 4  Try Importing segmentation_models_pytorch
 If not installed, a minimal fallback U-Net is created.
 **The fallback exists only to allow forward-pass testing.**
 Install smp before any training run:
 ```
 pip install segmentation-models-pytorch
 ```

In [14]:
SMP_AVAILABLE = False
try:
    import segmentation_models_pytorch as smp
    SMP_AVAILABLE = True
    print(f"segmentation_models_pytorch available — version {smp.__version__}")
except ImportError:
    print(
        "WARNING: segmentation_models_pytorch is NOT installed.\n"
        "         A minimal fallback U-Net will be created for forward-pass testing ONLY.\n"
        "         Install before training:  pip install segmentation-models-pytorch\n"
        "         Fallback does NOT have ImageNet-pretrained encoder weights."
    )

segmentation_models_pytorch available — version 0.5.0


# 5a  EfficientNet-B0 U-Net via smp (preferred path)


In [16]:
if SMP_AVAILABLE:
    model = smp.Unet(
        encoder_name   ="efficientnet-b0",
        encoder_weights="imagenet",
        in_channels    =IN_CHANNELS,
        classes        =NUM_CLASSES,
        activation     =None,       # output raw logits — sigmoid applied in loss
    )
    MODEL_NAME = "EfficientNet-B0 U-Net (smp)"
    print(f"Model created: {MODEL_NAME}")

Model created: EfficientNet-B0 U-Net (smp)


# 6  Move Model to Device

In [17]:
model = model.to(DEVICE)
print(f"Model on device: {DEVICE}")


Model on device: cuda


# 7  Parameter Count

In [18]:
def count_parameters(m: nn.Module):
    total     = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

total_params, trainable_params = count_parameters(model)

print("=" * 52)
print(f"  Model name         : {MODEL_NAME}")
print(f"  Device             : {DEVICE}")
print(f"  Total parameters   : {total_params:,}")
print(f"  Trainable params   : {trainable_params:,}")
print(f"  Frozen params      : {total_params - trainable_params:,}")
print("=" * 52)

  Model name         : EfficientNet-B0 U-Net (smp)
  Device             : cuda
  Total parameters   : 6,251,469
  Trainable params   : 6,251,469
  Frozen params      : 0


# 8  Dummy Forward Pass

 Input shape  : [2, 3, 384, 384]
 Expected out : [2, 1, 384, 384]  (raw logits)

In [19]:
model.eval()
with torch.no_grad():
    dummy_input  = torch.randn(2, IN_CHANNELS, IMG_SIZE, IMG_SIZE, device=DEVICE)
    dummy_output = model(dummy_input)

print(f"Input  shape : {dummy_input.shape}")
print(f"Output shape : {dummy_output.shape}")
print(f"Output dtype : {dummy_output.dtype}")
print(f"Output min   : {dummy_output.min().item():.4f}")
print(f"Output max   : {dummy_output.max().item():.4f}")

assert dummy_output.shape == (2, NUM_CLASSES, IMG_SIZE, IMG_SIZE), (
    f"Unexpected output shape: {dummy_output.shape}. "
    f"Expected (2, {NUM_CLASSES}, {IMG_SIZE}, {IMG_SIZE})."
)
print("Forward pass shape assertion passed.")

# Confirm output is raw logits: values should span across 0 (not squashed to [0,1])
# This is a heuristic check — logits from an untrained model will straddle 0.
assert dummy_output.min().item() < 0.0 or dummy_output.max().item() > 1.0, (
    "WARNING: output looks like it may have sigmoid applied. "
    "Ensure activation=None in smp.Unet."
)
print("Logit range check passed (no sigmoid detected).")

Input  shape : torch.Size([2, 3, 384, 384])
Output shape : torch.Size([2, 1, 384, 384])
Output dtype : torch.float32
Output min   : -10.0255
Output max   : 4.9995
Forward pass shape assertion passed.
Logit range check passed (no sigmoid detected).


# 9  Save Architecture Summary

In [20]:
summary_lines = [
    "=" * 60,
    "TONGUE SEGMENTATION MODEL ARCHITECTURE SUMMARY",
    "=" * 60,
    f"Model name         : {MODEL_NAME}",
    f"Device             : {DEVICE}",
    f"Input channels     : {IN_CHANNELS}",
    f"Output classes     : {NUM_CLASSES}",
    f"Input size         : {IMG_SIZE} x {IMG_SIZE}",
    f"Output activation  : None (raw logits — sigmoid in loss)",
    f"Loss function      : BCEWithLogitsLoss (applied during training)",
    f"Total parameters   : {total_params:,}",
    f"Trainable params   : {trainable_params:,}",
    f"Frozen params      : {total_params - trainable_params:,}",
    "-" * 60,
    "Dummy forward pass:",
    f"  Input  : {list(dummy_input.shape)}",
    f"  Output : {list(dummy_output.shape)}",
    f"  Out min: {dummy_output.min().item():.4f}",
    f"  Out max: {dummy_output.max().item():.4f}",
    "-" * 60,
    "smp available      : " + str(SMP_AVAILABLE),
    "=" * 60,
]

if SMP_AVAILABLE:
    # Append encoder name
    summary_lines.append(f"Encoder            : efficientnet-b0")
    summary_lines.append(f"Encoder weights    : imagenet")
    summary_lines.append(f"Decoder            : U-Net")

summary_text = "\n".join(summary_lines)
print(summary_text)

with open(ARCH_SUMMARY_PATH, "w", encoding="utf-8") as f:
    f.write(summary_text + "\n")

print(f"\nArchitecture summary saved -> {ARCH_SUMMARY_PATH}")
assert ARCH_SUMMARY_PATH.exists(), "ERROR: architecture summary file was not created."


TONGUE SEGMENTATION MODEL ARCHITECTURE SUMMARY
Model name         : EfficientNet-B0 U-Net (smp)
Device             : cuda
Input channels     : 3
Output classes     : 1
Input size         : 384 x 384
Output activation  : None (raw logits — sigmoid in loss)
Loss function      : BCEWithLogitsLoss (applied during training)
Total parameters   : 6,251,469
Trainable params   : 6,251,469
Frozen params      : 0
------------------------------------------------------------
Dummy forward pass:
  Input  : [2, 3, 384, 384]
  Output : [2, 1, 384, 384]
  Out min: -10.0255
  Out max: 4.9995
------------------------------------------------------------
smp available      : True
Encoder            : efficientnet-b0
Encoder weights    : imagenet
Decoder            : U-Net

Architecture summary saved -> D:\DIABETES\Segmentation_Dataset\Segmentation_Branch_Outputs\07_reports\05_model_architecture_summary.txt


# Notebook Complete

 | Output | Location |
 |--------|----------|
 | Architecture summary | `07_reports/05_model_architecture_summary.txt` |

 ### What was confirmed:
 - Model outputs **raw logits** — no sigmoid applied internally.
 - Output shape is `[B, 1, 384, 384]` — compatible with `BCEWithLogitsLoss`.
 - Forward pass runs without error on the configured device.

 ### What was NOT done:
 - No training.
 - No loss computation.
 - No optimizer definition.
 - No data was passed through the model (only a dummy random tensor).

 **Next step:** define training loop (future notebook).